In [2]:
#Imports
from langchain.chat_models import ChatOpenAI
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.schema.document import Document
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
import os
from dotenv import load_dotenv

In [3]:
#Load API key
load_dotenv(dotenv_path=".env")
openai_api_key=os.getenv("OPENAI_API_KEY")

#Initialize LLM
llm= ChatOpenAI(model="gpt-4o-mini", temperature=0)

C:\Users\mdfai\AppData\Local\Temp\ipykernel_11072\3725230459.py:6: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm= ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [7]:
#Create Vector DB (Retriever)
with open("sample.txt","r", encoding="utf-8")as f:
    text_data= f.read()

#Split the text into chunks
splitter= CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
chunks= splitter.split_text(text_data)
embedding= OpenAIEmbeddings()
documents= [Document(page_content=chunk) for chunk in chunks]

# 5. Compress the documents using LLMChainExtractor
compressor= LLMChainExtractor.from_llm(llm)
compressed_docs= compressor.compress_documents(documents, query="Summarize the key points")

print("Compressed Summary:")
for doc in compressed_docs:
    print("-", doc.page_content)

#Now ask a question about the compressed content using a custom LLMChain
qa_prompt= PromptTemplate.from_template(
    "Given the context below, answer the question:\n\nContext:\n{context}\n\nQuestion: {question}"
)
qa_chain=LLMChain(llm=llm, prompt=qa_prompt)

response= qa_chain.invoke({
    "context":"\n".join([doc.page_content for doc in compressed_docs]),
    "question":"What is the main idea of the document?"
})
print("\n💡 Answer to your question:")
print(response["text"])

Compressed Summary:
- LangChain is a framework for building applications with LLMs. LangChain was created by Harrison Chase. LangChain supports RAG, agents, memory, tools, and more. It’s commonly used in chatbots, document Q&A, and AI workflow.

💡 Answer to your question:
The main idea of the document is to provide an overview of LangChain, a framework designed for building applications that utilize large language models (LLMs). It highlights the key features of LangChain, such as its support for retrieval-augmented generation (RAG), agents, memory, and tools, as well as its common applications in chatbots, document question-and-answer systems, and AI workflows.
